# Feynman Eval — Explanation Quality & Hallucination Rate

Runs the agent end-to-end on a fixed topic, scores each explanation against
the Feynman rubric using Gemini, and checks citation grounding via the vector store.

**Target thresholds**
| Metric | Target |
|---|---|
| Feynman score (1–5) | ≥ 3.8 avg |
| Hallucination rate | < 5% |

In [ ]:
import sys
sys.path.insert(0, '..')

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from feynman.agent.nodes import research_node, assess_node, explain_node, MODEL
from feynman.eval.citation_checker import check_citations
from google import genai
from google.genai import types
import os, json

TOPIC = 'Vision Transformers'
SIMULATED_BACKGROUND = 'I know Python and have trained basic CNNs. Never used attention.'

key = os.getenv('GOOGLE_API_KEY')
print(f'Model:   {MODEL}')

client = genai.Client(api_key=key)
print('Setup complete.')

## Step 1 — Research (fetch papers, build index)

In [9]:
state = {'topic': TOPIC, 'user_message': TOPIC, 'phase': 'start'}
state = {**state, **research_node(state)}

print(f"Papers fetched:    {len(state['papers'])}")
print(f"Concepts ordered:  {state['concepts_ordered']}")
print(f"\nAssessment Qs:\n{state['agent_response'][:400]}...")

2026-05-14 20:10:10 [info     ] research_start                 topic='Vision Transformers'
2026-05-14 20:10:10 [info     ] arxiv_search_done              query='Vision Transformers' returned=15
2026-05-14 20:10:11 [warning  ] rate_limited                   attempt=0 wait_s=1.0
2026-05-14 20:10:12 [warning  ] rate_limited                   attempt=1 wait_s=2.0
2026-05-14 20:10:14 [warning  ] rate_limited                   attempt=2 wait_s=4.0
2026-05-14 20:10:18 [debug    ] enriched                       citations=4 paper_id=2210.15722v1
2026-05-14 20:10:18 [warning  ] rate_limited                   attempt=0 wait_s=1.0
2026-05-14 20:10:19 [warning  ] rate_limited                   attempt=1 wait_s=2.0
2026-05-14 20:10:21 [warning  ] rate_limited                   attempt=2 wait_s=4.0
2026-05-14 20:10:25 [warning  ] rate_limited                   attempt=3 wait_s=8.0
2026-05-14 20:10:33 [error    ] ss_max_retries_exceeded        url=https://api.semanticscholar.org/graph/v1/paper/arXiv:2

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6347.16it/s]


2026-05-14 20:12:38 [info     ] vector_store_init              device=mps model=all-MiniLM-L6-v2
2026-05-14 20:12:39 [info     ] pdf_parsed                     chunks=44 paper_id=2205.10063v1 sections=23
2026-05-14 20:12:39 [info     ] paper_indexed                  chunks=44 paper_id=2205.10063v1 total=44
2026-05-14 20:12:39 [info     ] indexed                        chunks=44 title='Uniform Masking: Enabling MAE Pre-training for Pyramid-based'
2026-05-14 20:12:40 [info     ] pdf_parsed                     chunks=133 paper_id=2205.04675v2 sections=103
2026-05-14 20:12:40 [info     ] paper_indexed                  chunks=133 paper_id=2205.04675v2 total=177
2026-05-14 20:12:40 [info     ] indexed                        chunks=133 title='Spatial Monitoring and Insect Behavioural Analysis Using Com'
2026-05-14 20:12:41 [info     ] pdf_parsed                     chunks=34 paper_id=2501.00142v1 sections=12
2026-05-14 20:12:41 [info     ] paper_indexed                  chunks=34 paper_id=250

## Step 2 — Assess (infer user level)

In [3]:
state['user_message'] = SIMULATED_BACKGROUND
state = {**state, **assess_node(state)}
print(f"User level:       {state['user_level']}")
print(f"Starting concept: {state['current_concept']}")

2026-05-14 20:00:39 [info     ] assessed                       starting='Self-Attention Mechanisms'
User level:       intermediate
Starting concept: Self-Attention Mechanisms


## Step 3 — Explain + Score each concept

In [5]:
FEYNMAN_RUBRIC = """
Score this explanation on the Feynman teaching rubric (1–5):

1 = Jargon-heavy, no analogy, assumes prior knowledge
2 = Some plain language but key ideas unclear
3 = Clear explanation with some analogy
4 = Simple language, concrete analogy, builds on prior knowledge
5 = Exceptional clarity, perfect analogy, cites source, one precise Socratic question

Explanation to score:
{explanation}

Reply with ONLY valid JSON: {{"score": 4, "reason": "one sentence"}}
"""

def feynman_score(explanation: str) -> dict:
    resp = client.models.generate_content(
        model=MODEL,
        contents=FEYNMAN_RUBRIC.format(explanation=explanation[:2000]),
        config=types.GenerateContentConfig(temperature=0.1, max_output_tokens=256),
    )
    text = resp.text.strip()
    import re
    m = re.search(r'```(?:json)?\s*([\s\S]*?)```', text)
    if m: text = m.group(1)
    try:
        return json.loads(text)
    except Exception:
        return {'score': 0, 'reason': f'parse error: {text[:100]}'}

results = []
max_concepts = len(state['concepts_ordered'])

for i in range(max_concepts):
    state['reexplain'] = False
    explain_result = explain_node(state)
    
    if explain_result.get('phase') == 'summarizing':
        print('All concepts covered — stopping.')
        break
    
    state = {**state, **explain_result}
    explanation = state['agent_response']
    concept = state['current_concept']
    
    # Feynman score
    score_data = feynman_score(explanation)
    
    # Citation grounding
    citation = check_citations(explanation, state['vector_store'])
    
    results.append({
        'concept': concept,
        'feynman_score': score_data.get('score', 0),
        'reason': score_data.get('reason', ''),
        'grounding_rate': citation.grounding_rate,
        'hallucination_rate': citation.hallucination_rate,
        'explanation_snippet': explanation[:200],
    })
    
    print(f"[{i+1}/{max_concepts}] {concept}")
    print(f"  Feynman score:      {score_data.get('score')}/5 — {score_data.get('reason')}")
    print(f"  Hallucination rate: {citation.hallucination_rate:.1%}")
    print()
    
    # Simulate user understanding to advance
    covered = list(state.get('concepts_covered', []))
    if concept not in covered:
        covered.append(concept)
    state['concepts_covered'] = covered

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 9.305195493s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.goo

## Step 4 — Results summary

In [ ]:
import statistics

scores = [r['feynman_score'] for r in results if r['feynman_score'] > 0]
hall_rates = [r['hallucination_rate'] for r in results]

avg_score = statistics.mean(scores) if scores else 0
avg_hall = statistics.mean(hall_rates) if hall_rates else 0

print('='*60)
print(f'TOPIC:                  {TOPIC}')
print(f'CONCEPTS EVALUATED:     {len(results)}')
print(f'AVG FEYNMAN SCORE:      {avg_score:.2f}/5  (target ≥ 3.8)')
print(f'AVG HALLUCINATION RATE: {avg_hall:.1%}  (target < 5%)')
print('='*60)
print()
print('| Concept | Feynman Score | Hallucination Rate |')
print('|---|---|---|')
for r in results:
    print(f"| {r['concept']} | {r['feynman_score']}/5 | {r['hallucination_rate']:.1%} |")
print()
print('Copy the table above into README.md under ## Eval Results')